# Text Summarization with TF-IDF
Extractive summarization: select the most important sentences based on TF-IDF scores.

**Approach:**
1. Tokenize document into sentences
2. Remove stop words (English: sklearn built-in, Indonesian: Sastrawi)
3. Compute TF-IDF for each sentence
4. Score each sentence by summing its word TF-IDF values
5. Select top-N sentences as the summary

## 0. Install Dependencies

In [ ]:
!pip install -q nltk scikit-learn matplotlib Sastrawi

## 1. Import Libraries

In [ ]:
import nltk
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from nltk.tokenize import sent_tokenize
from sklearn.feature_extraction.text import TfidfVectorizer
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
print('All libraries loaded.')

## 2. Define Documents

In [ ]:
# Document 1: English
doc_en = """
Manchester City makes history by winning Club World Cup

Manchester City capped off its incredible year with yet another trophy, dismantling Fluminense 4-0 to win the Club World Cup on Friday.

Having already won the Premier League, Champions League, FA Cup and Super Cup, Pep Guardiola's side now boasts five trophies this calendar year, becoming the first English club to ever hold all those titles simultaneously.

The final piece of the jigsaw came on a highly charged night in Saudi Arabia as Manchester City outclassed its Brazilian opponents.

“We’ve shown over the past 12 months we are the best team in the world. Our results prove that and the consistency we have managed has been amazing,” club captain Kyle Walker said after the game, per Sky Sports.

“To win these five trophies – for me, the five biggest prizes available to us – is incredible. I am so proud to have been a part of this and I can honestly say it’s an honour to play alongside these players. I couldn’t ask for better teammates.”

It took just 40 seconds for Manchester City to take the lead.

Brazilian left-back Marcelo miscued a pass in the opening exchanges which let Nathan Aké free to shoot from distance. The defender’s effort cannoned back off the post but forward Julián Álvarez was in the right place to turn the rebound into the net with his chest.

City continued to look dangerous and doubled its lead before the break after Phil Foden's attempted cross was deflected into his own net by Fluminense defender Nino.

Foden then got on the scoresheet himself in the 72nd minute after a prodding home from close range.

The rout was completed in the 88th minute when Álvarez capped off a brilliant performance with a clinical finish into the far corner.

“As a manager it is what I am most proud of; that we are always there. No matter how much we win, no matter what trophies we lift, we are there again to fight for the next one,” City boss Guardiola said after the match, according to Sky Sports.

“To win the Treble was truly special, but to win two more trophies and now hold these five major titles shows the unique mentality of this team, of the Club and its fans.”
"""
print('Document 1 (English) defined.')

In [ ]:
# Document 2: Indonesian
doc_id = """
Jakarta: Badan Pengelola Investasi Daya Anagata Nusantara (BPI Danantara) siap mengawal realisasi investasi yang telah disepakati dengan Qatar. Kesepakatan antara Indonesia dan Qatar merupakan buah dari kunjungan resmi Presiden Prabowo Subianto ke Doha.

Pemerintah Republik Indonesia dan Pemerintah Qatar menggelar diskusi untuk menyepakati kemitraan strategis (co-partnership) dalam pengelolaan dana investasi untuk Indonesia yang akan berfokus di berbagai sektor pembangunan.

Salah satu hasil utama dari kunjungan tersebut adalah untuk membentuk dana investasi bersama senilai USD4 miliar. Dana ini akan difokuskan pada pengembangan berbagai sektor di antaranya termasuk tapi tidak terbatas pada hilirisasi industri, energi terbarukan, dan fasilitas kesehatan di Indonesia.

"Kami menyambut baik kepercayaan yang diberikan oleh Pemerintah Qatar melalui pembentukan dana bersama ini," kata CEO Danantara Indonesia Rosan Perkasa Roeslani dalam keterangan tertulis, Selasa, 15 April 2025.

Presiden Prabowo menyampaikan masing-masing negara akan berkontribusi sebesar USD2 miliar dalam dana tersebut. Dana itu akan dikelola oleh BPI Danantara bersama dengan Qatar Investment Authority (QIA) dalam co-partnership.

Dana tersebut akan difokuskan pada peluang investasi di berbagai sektor strategis, antara lain hilirisasi, kesehatan, energi terbarukan, teknologi, serta sektor-sektor lain yang dipandang relevan oleh pengelola dana.

"Danantara Indonesia siap menjalankan mandat tersebut dengan menerapkan tata kelola investasi yang prudent, transparan, dan berorientasi pada hasil. Fokus kami adalah memastikan bahwa setiap proyek yang didanai memberikan dampak strategis dan berkelanjutan bagi perekonomian nasional," ujar Rosan.

Lebih lanjut, Rosan menegaskan, kolaborasi ini menjadi bukti kepercayaan dunia internasional terhadap kapasitas kelembagaan Indonesia dalam mengelola investasi berskala besar.

"Kemitraan ini merupakan langkah konkret dalam membangun kepercayaan dengan mitra global strategis seperti Qatar. Ini menunjukkan bahwa Indonesia tidak hanya menjadi tujuan investasi, tetapi juga memiliki kapasitas kelembagaan yang mumpuni untuk mengelola investasi secara profesional dan akuntabel," ungkapnya.

Inisiatif co-partnership dan perluasan kerja sama strategis ini diharapkan tidak hanya memperkuat hubungan diplomatik kedua negara, tetapi juga memberikan kontribusi nyata terhadap percepatan pembangunan ekonomi dan peningkatan kesejahteraan masyarakat Indonesia.
"""
print('Document 2 (Indonesian) defined.')

## 3. TF-IDF Summarizer Function

In [ ]:
def tfidf_summarize(text, lang='en', top_n=3):
    """
    Extractive summarization using TF-IDF sentence scoring.

    Parameters
    ----------
    text   : str  - raw input document
    lang   : str  - 'en' for English, 'id' for Indonesian
    top_n  : int  - number of sentences to include in summary

    Returns
    -------
    summary         : str  - extracted summary
    sentence_scores : list - list of (sentence, score) tuples
    """
    # Step 1: Sentence tokenization
    sentences = sent_tokenize(text.strip())
    sentences = [s.strip() for s in sentences if len(s.strip()) > 10]

    # Step 2: Stop word removal
    if lang == 'id':
        factory = StopWordRemoverFactory()
        remover = factory.create_stop_word_remover()
        cleaned = [remover.remove(s) for s in sentences]
        vectorizer = TfidfVectorizer()           # Sastrawi already handled stop words
    else:
        cleaned = sentences
        vectorizer = TfidfVectorizer(stop_words='english')

    # Step 3: Build TF-IDF matrix (sentences x words)
    tfidf_matrix = vectorizer.fit_transform(cleaned)

    # Step 4: Score each sentence (sum of its TF-IDF values)
    scores = np.array(tfidf_matrix.sum(axis=1)).flatten()

    # Step 5: Select top-N sentences, preserve original order
    top_indices = np.argsort(scores)[::-1][:top_n]
    top_indices_ordered = sorted(top_indices)

    summary = ' '.join([sentences[i] for i in top_indices_ordered])
    sentence_scores = list(zip(sentences, scores))

    return summary, sentence_scores


print('Summarizer function ready.')

## 4. Summarize Document 1 (English)

In [ ]:
summary_en, scores_en = tfidf_summarize(doc_en, lang='en', top_n=3)

print('=== SUMMARY (English, top 3 sentences) ===')
print(summary_en)
print()
print('=== SENTENCE SCORES ===')
scores_df_en = pd.DataFrame(scores_en, columns=['Sentence', 'Score'])
scores_df_en['Sentence'] = scores_df_en['Sentence'].str[:80] + '...'
print(scores_df_en.sort_values('Score', ascending=False).to_string(index=False))

## 5. Summarize Document 2 (Indonesian)

In [ ]:
summary_id, scores_id = tfidf_summarize(doc_id, lang='id', top_n=3)

print('=== RINGKASAN (Indonesian, top 3 kalimat) ===')
print(summary_id)
print()
print('=== SKOR TIAP KALIMAT ===')
scores_df_id = pd.DataFrame(scores_id, columns=['Kalimat', 'Skor'])
scores_df_id['Kalimat'] = scores_df_id['Kalimat'].str[:80] + '...'
print(scores_df_id.sort_values('Skor', ascending=False).to_string(index=False))

## 6. Visualization: Sentence Scores

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# English
labels_en = [f'S{i+1}' for i in range(len(scores_en))]
vals_en   = [s for _, s in scores_en]
top_idx_en = np.argsort(vals_en)[::-1][:3]
colors_en  = ['#e74c3c' if i in top_idx_en else '#aab7b8' for i in range(len(vals_en))]

axes[0].bar(labels_en, vals_en, color=colors_en)
axes[0].set_title('Document 1 (English)\nSentence TF-IDF Scores (red = selected)')
axes[0].set_xlabel('Sentence')
axes[0].set_ylabel('TF-IDF Score (sum)')

# Indonesian
labels_id = [f'K{i+1}' for i in range(len(scores_id))]
vals_id   = [s for _, s in scores_id]
top_idx_id = np.argsort(vals_id)[::-1][:3]
colors_id  = ['#e74c3c' if i in top_idx_id else '#aab7b8' for i in range(len(vals_id))]

axes[1].bar(labels_id, vals_id, color=colors_id)
axes[1].set_title('Dokumen 2 (Indonesian)\nSkor TF-IDF Tiap Kalimat (merah = terpilih)')
axes[1].set_xlabel('Kalimat')
axes[1].set_ylabel('TF-IDF Score (sum)')

plt.tight_layout()
plt.show()

## Notes

**How it works:**
- Each sentence is treated as a separate document in the TF-IDF model.
- A sentence scores high when it contains words that are frequent within that sentence but rare across others — meaning it carries unique, important information.
- `top_n=3` selects the 3 highest-scoring sentences, returned in their original order so the summary reads naturally.

**Stop word handling:**
- English: `TfidfVectorizer(stop_words='english')` removes common words automatically.
- Indonesian: Sastrawi `StopWordRemover` handles Bahasa Indonesia stop words before vectorization.

**Limitation:** TF-IDF is position-unaware and does not model sentence relationships. For better results, consider TextRank or adding a positional bonus to the scoring.